# 🎬 Agente Experto en Cine (Tarantino) — RAG + LangGraph + Gemini 2.5 Flash

In [1]:
!pip install -q langchain langchain-community langchain-chroma langgraph langchain-google-genai pypdf chromadb langgraph python-dotenv google-generativeai
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


## Paso 1 — Configuración y API Key

In [2]:
import os
from google.colab import userdata

# -- API KEY (guardada en Secrets de Colab, icono 🔑) ----------
try:
    GEMINI_API_KEY = userdata.get("Uja-JAC")
except Exception:
    GEMINI_API_KEY = os.environ.get("Uja-JAC", "")

if not GEMINI_API_KEY:
    raise ValueError(
        "❌ API key no encontrada.\n"
        "Ve a Secrets (🔑) en el panel izquierdo de Colab\n"
        "y añade la clave con el nombre: GEMINI_API_KEY"
    )

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# -- Parámetros globales del proyecto --------------------------
TEMA_EXPERTO    = "Cine y películas de Quentin Tarantino"
PDF_DIR         = "/content/pdfs"
CHROMA_DIR      = "/content/chroma_db"
COLLECTION_NAME = "tarantino_expert"
CHUNK_SIZE      = 1500      # caracteres por chunk
CHUNK_OVERLAP   = 200      # solapamiento entre chunks
TOP_K           = 5        # chunks recuperados por consulta
EMBEDDING_MODEL = "models/embedding-001"
LLM_MODEL       = "gemini-2.5-flash-lite"

print(f"✅ Configuración lista")
print(f"   Tema       : {TEMA_EXPERTO}")
print(f"   LLM        : {LLM_MODEL}")
print(f"   Embeddings : {EMBEDDING_MODEL}")
print(f"   Chunk size : {CHUNK_SIZE} chars  |  overlap: {CHUNK_OVERLAP}")
print(f"   Top-K      : {TOP_K}")


✅ Configuración lista
   Tema       : Cine y películas de Quentin Tarantino
   LLM        : gemini-2.5-flash-lite
   Embeddings : models/embedding-001
   Chunk size : 1500 chars  |  overlap: 200
   Top-K      : 5


## Paso 2 — Subir documentos PDF

In [3]:
import os, shutil
from google.colab import files

os.makedirs(PDF_DIR, exist_ok=True)

print("📁 Sube tus PDFs aquí (mínimo 3 documentos):")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = os.path.join(PDF_DIR, filename)
    shutil.move(filename, dest)
    print(f"   ✓ {filename} → {dest}")

pdf_files = [f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf")]
print(f"\n📚 Total PDFs disponibles: {len(pdf_files)}")
for f in pdf_files:
    kb = os.path.getsize(os.path.join(PDF_DIR, f)) / 1024
    print(f"   • {f}  ({kb:.1f} KB)")

if len(pdf_files) < 3:
    print("\n⚠️  Se recomiendan al menos 3 PDFs para el MVP.")

📁 Sube tus PDFs aquí (mínimo 3 documentos):


Saving Death_Proof.pdf to Death_Proof.pdf
Saving Django_Unchained.pdf to Django_Unchained.pdf
Saving Inglourious_Basterds.pdf to Inglourious_Basterds.pdf
Saving Jackie_Brown.pdf to Jackie_Brown.pdf
Saving Kill_Bill__Volumen_1.pdf to Kill_Bill__Volumen_1.pdf
Saving Kill_Bill__Volumen_2.pdf to Kill_Bill__Volumen_2.pdf
Saving Once_Upon_a_Time_in_Hollywood.pdf to Once_Upon_a_Time_in_Hollywood.pdf
Saving Pulp_Fiction.pdf to Pulp_Fiction.pdf
Saving Reservoir_Dogs.pdf to Reservoir_Dogs.pdf
Saving The_Hateful_Eight.pdf to The_Hateful_Eight.pdf
   ✓ Death_Proof.pdf → /content/pdfs/Death_Proof.pdf
   ✓ Django_Unchained.pdf → /content/pdfs/Django_Unchained.pdf
   ✓ Inglourious_Basterds.pdf → /content/pdfs/Inglourious_Basterds.pdf
   ✓ Jackie_Brown.pdf → /content/pdfs/Jackie_Brown.pdf
   ✓ Kill_Bill__Volumen_1.pdf → /content/pdfs/Kill_Bill__Volumen_1.pdf
   ✓ Kill_Bill__Volumen_2.pdf → /content/pdfs/Kill_Bill__Volumen_2.pdf
   ✓ Once_Upon_a_Time_in_Hollywood.pdf → /content/pdfs/Once_Upon_a_Time_in

## Paso 3 — Preprocesamiento y chunking

In [4]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def clean_text(text: str) -> str:
    """Limpieza básica: normaliza espacios y elimina ruido de OCR."""
    text = re.sub(r"\s+",   " ",  text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[^\S\n]+", " ", text)
    text = re.sub(r"\.{4,}", "...", text)
    return text.strip()


def load_pdfs(folder: str):
    """Carga todos los PDFs de una carpeta página a página."""
    docs = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        loader = PyPDFLoader(os.path.join(folder, fname))
        pages  = loader.load()
        docs.extend(pages)
        print(f"   📄 {fname} → {len(pages)} páginas")
    return docs


print("PASO 2 — Cargando y preprocesando documentos...")
print("=" * 55)

raw_docs = load_pdfs(PDF_DIR)

# Limpiar contenido y filtrar páginas vacías
for doc in raw_docs:
    doc.page_content = clean_text(doc.page_content)
raw_docs = [d for d in raw_docs if len(d.page_content.strip()) > 50]

# Chunking con ventana deslizante
splitter = RecursiveCharacterTextSplitter(
    chunk_size      = CHUNK_SIZE,
    chunk_overlap   = CHUNK_OVERLAP,
    separators      = ["\n\n", "\n", ". ", " ", ""],
    length_function = len,
)
chunks = splitter.split_documents(raw_docs)

# Añadir metadatos de identificación
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"]   = i
    chunk.metadata["char_count"] = len(chunk.page_content)

print(f"\n📊 Resultados del chunking:")
print(f"   • Páginas válidas  : {len(raw_docs)}")
print(f"   • Total chunks     : {len(chunks)}")
print(f"   • Chars promedio   : {sum(len(c.page_content) for c in chunks) // len(chunks)}")
print(f"   • Chunk más corto  : {min(len(c.page_content) for c in chunks)} chars")
print(f"   • Chunk más largo  : {max(len(c.page_content) for c in chunks)} chars")
print(f"\n📝 Ejemplo — chunk #0:")
print(f"   Fuente : {chunks[0].metadata.get('source','N/A').split('/')[-1]}")
print(f"   Página : {chunks[0].metadata.get('page','N/A')}")
print(f"   Texto  : {chunks[0].page_content[:250]}...")

PASO 2 — Cargando y preprocesando documentos...
   📄 Death_Proof.pdf → 5 páginas
   📄 Django_Unchained.pdf → 6 páginas
   📄 Inglourious_Basterds.pdf → 9 páginas
   📄 Jackie_Brown.pdf → 5 páginas
   📄 Kill_Bill__Volumen_1.pdf → 8 páginas
   📄 Kill_Bill__Volumen_2.pdf → 7 páginas
   📄 Once_Upon_a_Time_in_Hollywood.pdf → 23 páginas
   📄 Pulp_Fiction.pdf → 17 páginas
   📄 Reservoir_Dogs.pdf → 7 páginas
   📄 The_Hateful_Eight.pdf → 5 páginas

📊 Resultados del chunking:
   • Páginas válidas  : 91
   • Total chunks     : 218
   • Chars promedio   : 1150
   • Chunk más corto  : 78 chars
   • Chunk más largo  : 1499 chars

📝 Ejemplo — chunk #0:
   Fuente : Death_Proof.pdf
   Página : 0
   Texto  : Death Proof Título Death Proof (España) A prueba de muerte (Hispanoamérica) Ficha técnica Dirección Quentin Tarantino Producción Quentin Tarantino Elizabeth Avellán Erica Steinberg Robert Rodriguez Guion Quentin Tarantino Música David Arnold Fotograf...


## Paso 4 — ChromaDB + Gemini Embeddings:

In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — PASO 3: ChromaDB + Gemini Embeddings              ║
# ║  con reintentos automáticos para el rate limit               ║
# ╚══════════════════════════════════════════════════════════════╝

import time
import math
import chromadb
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

os.makedirs(CHROMA_DIR, exist_ok=True)

# ── EMBEDDINGS CON RETRY MANUAL ────────────────────────────────
# El tier gratuito permite 100 requests/min.
# Indexamos en lotes pequeños con pausa entre ellos.

BATCH_SIZE   = 20    # chunks por lote (bien por debajo del límite)
WAIT_SECONDS = 15    # pausa entre lotes (segundos)
MAX_RETRIES  = 5     # reintentos por lote si hay error 429

embeddings = GoogleGenerativeAIEmbeddings(
    model     = "gemini-embedding-001",
    task_type = "retrieval_document",
)

def embed_with_retry(batch, attempt=1):
    """Embebe un lote con backoff exponencial si hay rate limit."""
    try:
        return embeddings.embed_documents([c.page_content for c in batch])
    except Exception as e:
        if "429" in str(e) and attempt <= MAX_RETRIES:
            wait = WAIT_SECONDS * attempt          # 15s, 30s, 45s...
            print(f"   ⏳ Rate limit — esperando {wait}s (intento {attempt}/{MAX_RETRIES})...")
            time.sleep(wait)
            return embed_with_retry(batch, attempt + 1)
        raise

print("PASO 3 — Indexando en ChromaDB con Gemini Embeddings...")
print("=" * 55)
print(f"   Chunks totales : {len(chunks)}")
print(f"   Tamaño de lote : {BATCH_SIZE}")
print(f"   Pausa entre lotes: {WAIT_SECONDS}s")
print(f"   Lotes estimados: {math.ceil(len(chunks)/BATCH_SIZE)}\n")

# ── INDEXACIÓN POR LOTES ───────────────────────────────────────
# Creamos el vectorstore con el primer lote y luego añadimos el resto.

batches     = [chunks[i:i+BATCH_SIZE] for i in range(0, len(chunks), BATCH_SIZE)]
vectorstore = None

for idx, batch in enumerate(batches):
    print(f"   🔄 Lote {idx+1}/{len(batches)}  ({len(batch)} chunks)...", end=" ")

    if vectorstore is None:
        # Primer lote: crea la colección
        vectorstore = Chroma.from_documents(
            documents         = batch,
            embedding         = embeddings,
            collection_name   = COLLECTION_NAME,
            persist_directory = CHROMA_DIR,
        )
    else:
        # Lotes siguientes: añade a la colección existente
        vectorstore.add_documents(batch)

    print(f"✓  (total: {vectorstore._collection.count()})")

    # Pausa entre lotes para respetar el rate limit
    if idx < len(batches) - 1:
        time.sleep(WAIT_SECONDS)

vectorstore.persist()

print(f"\n✅ ChromaDB indexada correctamente")
print(f"   • Colección  : {COLLECTION_NAME}")
print(f"   • Documentos : {vectorstore._collection.count()}")
print(f"   • Persistida : {CHROMA_DIR}")

# ── VERIFICACIÓN DEL ÍNDICE ────────────────────────────────────
print("\n" + "─" * 55)
print("🔍 VERIFICACIÓN — consultas de prueba")
print("─" * 55)

retriever_test = vectorstore.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 3},
)

test_queries = [
    "Pulp Fiction argumento personajes",
    "premios Oscar nominaciones Tarantino",
    "Kill Bill reparto Uma Thurman",
]

for q in test_queries:
    results = retriever_test.invoke(q)
    print(f"\n  Query : \"{q}\"")
    for i, r in enumerate(results[:2]):
        src     = r.metadata.get("source", "N/A").split("/")[-1]
        page    = r.metadata.get("page", "?")
        preview = r.page_content[:110].replace("\n", " ")
        print(f"    [{i+1}] {src} p.{page} → \"{preview}...\"")

print("\n✅ Índice verificado. Listo para conectar el agente.")

PASO 3 — Indexando en ChromaDB con Gemini Embeddings...
   Chunks totales : 218
   Tamaño de lote : 20
   Pausa entre lotes: 15s
   Lotes estimados: 11

   🔄 Lote 1/11  (20 chunks)... ✓  (total: 1032)
   🔄 Lote 2/11  (20 chunks)... ✓  (total: 1052)
   🔄 Lote 3/11  (20 chunks)... ✓  (total: 1072)
   🔄 Lote 4/11  (20 chunks)... ✓  (total: 1092)
   🔄 Lote 5/11  (20 chunks)... ✓  (total: 1112)
   🔄 Lote 6/11  (20 chunks)... ✓  (total: 1132)
   🔄 Lote 7/11  (20 chunks)... ✓  (total: 1152)
   🔄 Lote 8/11  (20 chunks)... ✓  (total: 1172)
   🔄 Lote 9/11  (20 chunks)... ✓  (total: 1192)
   🔄 Lote 10/11  (20 chunks)... ✓  (total: 1212)
   🔄 Lote 11/11  (18 chunks)... ✓  (total: 1230)

✅ ChromaDB indexada correctamente
   • Colección  : tarantino_expert
   • Documentos : 1230
   • Persistida : /content/chroma_db

───────────────────────────────────────────────────────
🔍 VERIFICACIÓN — consultas de prueba
───────────────────────────────────────────────────────


/tmp/ipykernel_47047/1253963199.py:73: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()



  Query : "Pulp Fiction argumento personajes"
    [1] Pulp_Fiction.pdf p.13 → ". Consultado el 9 de diciembre de 2017. 18. Pulp Fiction - The facts (htt ps://www.youtube.com/wat ch?v=4Xjkwc..."
    [2] Pulp_Fiction.pdf p.13 → ". Consultado el 9 de diciembre de 2017. 18. Pulp Fiction - The facts (htt ps://www.youtube.com/wat ch?v=4Xjkwc..."

  Query : "premios Oscar nominaciones Tarantino"
    [1] Pulp_Fiction.pdf p.11 → "Categoría Receptor Resultado Mejor película Lawrence Bender Nominado13 Mejor dirección Quentin Tarantino Nomin..."
    [2] Pulp_Fiction.pdf p.11 → "Categoría Receptor Resultado Mejor película Lawrence Bender Nominado13 Mejor dirección Quentin Tarantino Nomin..."

  Query : "Kill Bill reparto Uma Thurman"
    [1] Kill_Bill__Volumen_2.pdf p.0 → "Kill Bill: Volumen 2 Ficha técnica Dirección Quentin Tarantino Producción Lawrence Bender Guion Quentin Tarant..."
    [2] Kill_Bill__Volumen_2.pdf p.0 → "Kill Bill: Volumen 2 Ficha técnica Dirección Quentin Tarantino Producción

## Paso 5 — System prompt

In [6]:
SYSTEM_PROMPT = """
Eres un asistente experto en el cine de Quentin Tarantino.

Tu conocimiento proviene EXCLUSIVAMENTE de los fragmentos de documentos que se te
proporcionan en cada consulta. Sigue estas reglas sin excepción:

1. CONTEXTO PRIMERO: Basa tus respuestas únicamente en los fragmentos recuperados.
   No uses conocimiento externo ni inventes información.

2. HONESTIDAD: Si los fragmentos no contienen información suficiente, dilo:
   "No tengo esa información en mi base de conocimiento."

3. CITAS: Cuando sea posible, menciona de qué documento proviene la información
   (ej.: "Según el documento X...").

4. TONO: Analítico, preciso y accesible. Puedes usar terminología cinematográfica
   pero explica los términos técnicos cuando sea necesario.

5. MEMORIA: Tienes acceso al historial de conversación. Mantén coherencia con
   respuestas anteriores y haz referencia a ellas cuando sea relevante.

6. IDIOMA: Responde siempre en español.

7. FORMATO: Usa listas y negritas cuando mejore la legibilidad.
""".strip()

print("✅ System Prompt configurado")
print("─" * 55)
print(SYSTEM_PROMPT)

✅ System Prompt configurado
───────────────────────────────────────────────────────
Eres un asistente experto en el cine de Quentin Tarantino.
 
Tu conocimiento proviene EXCLUSIVAMENTE de los fragmentos de documentos que se te
proporcionan en cada consulta. Sigue estas reglas sin excepción:
 
1. CONTEXTO PRIMERO: Basa tus respuestas únicamente en los fragmentos recuperados.
   No uses conocimiento externo ni inventes información.
 
2. HONESTIDAD: Si los fragmentos no contienen información suficiente, dilo:
   "No tengo esa información en mi base de conocimiento."
 
3. CITAS: Cuando sea posible, menciona de qué documento proviene la información
   (ej.: "Según el documento X...").
 
4. TONO: Analítico, preciso y accesible. Puedes usar terminología cinematográfica
   pero explica los términos técnicos cuando sea necesario.
 
5. MEMORIA: Tienes acceso al historial de conversación. Mantén coherencia con
   respuestas anteriores y haz referencia a ellas cuando sea relevante.
 
6. IDIOMA: Re

## Paso 6 — Agente LangGraph

In [7]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, BaseMessage
)
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# -- LLM -------------------------------------------------------
llm = ChatGoogleGenerativeAI(
    model          = LLM_MODEL,
    google_api_key = GEMINI_API_KEY,
    temperature    = 0.2,
    convert_system_message_to_human = True,   # requerido por Gemini
)

# -- Retriever (MMR para diversidad de resultados) -------------
retriever = vectorstore.as_retriever(
    search_type   = "mmr",
    search_kwargs = {"k": TOP_K, "fetch_k": TOP_K * 3},
)

# -- Estado del grafo ------------------------------------------
class AgentState(TypedDict):
    messages : Annotated[List[BaseMessage], add_messages]
    context  : str
    query    : str
    sources  : List[str]

# -- Nodo 1: RETRIEVE ------------------------------------------
def retrieve_node(state: AgentState) -> AgentState:
    """Recupera los chunks más relevantes de ChromaDB."""
    docs    = retriever.invoke(state["query"])
    parts   = []
    sources = []
    for i, doc in enumerate(docs, 1):
        src  = doc.metadata.get("source", "Desconocido").split("/")[-1]
        page = doc.metadata.get("page", "?")
        parts.append(f"[Fragmento {i} — {src}, pág. {page}]\n{doc.page_content}")
        sources.append(f"{src} (pág. {page})")
    context = "\n\n" + ("─" * 40 + "\n\n").join(parts)
    return {**state, "context": context, "sources": sources}

# -- Nodo 2: GENERATE ------------------------------------------
def generate_node(state: AgentState) -> AgentState:
    """Genera la respuesta combinando historial + contexto RAG."""
    augmented = (
        f"CONTEXTO RECUPERADO DE LA BASE DE CONOCIMIENTO:\n"
        f"{state['context']}\n\n"
        f"{'═' * 40}\n\n"
        f"PREGUNTA DEL USUARIO: {state['query']}"
    )
    messages_for_llm = [
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"][:-1],                       # historial previo
        HumanMessage(content=augmented),               # turno actual + contexto
    ]
    response = llm.invoke(messages_for_llm)
    return {**state, "messages": [AIMessage(content=response.content)]}

# -- Construcción del grafo ------------------------------------
builder = StateGraph(AgentState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

agent = builder.compile()

print("✅ Agente LangGraph compilado")
print("   Flujo: retrieve → generate → END")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Agente LangGraph compilado
   Flujo: retrieve → generate → END


## Paso 7 — Memoria de conversación

In [8]:
class ConversationMemory:
    """
    Gestiona el historial multi-turno del agente.
    Almacena pares (HumanMessage, AIMessage) y los inyecta
    en el estado del grafo en cada nueva consulta.
    """

    def __init__(self, max_turns: int = 10):
        self.history: List[BaseMessage] = []
        self.max_turns = max_turns

    def add_turn(self, human: str, ai: str):
        self.history.append(HumanMessage(content=human))
        self.history.append(AIMessage(content=ai))
        # Truncar para no superar el límite de contexto
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-(self.max_turns * 2):]

    def get(self) -> List[BaseMessage]:
        return list(self.history)

    def clear(self):
        self.history = []
        print("🗑  Historial de memoria limpiado.")

    def show(self):
        if not self.history:
            print("  (Sin historial todavía)")
            return
        for msg in self.history:
            role    = "👤 Usuario" if isinstance(msg, HumanMessage) else "🎬 Agente"
            preview = msg.content[:100].replace("\n", " ")
            print(f"  {role}: {preview}...")


# -- Función principal de consulta -----------------------------
def ask(question: str, verbose: bool = True) -> str:
    """
    Consulta el agente RAG con memoria.

    Args:
        question : Pregunta en lenguaje natural.
        verbose  : Mostrar pregunta, respuesta y fuentes.

    Returns:
        Respuesta del agente como string.
    """
    initial_state: AgentState = {
        "messages" : memory.get() + [HumanMessage(content=question)],
        "context"  : "",
        "query"    : question,
        "sources"  : [],
    }

    result = agent.invoke(initial_state)
    answer = result["messages"][-1].content

    memory.add_turn(question, answer)

    if verbose:
        sep = "─" * 60
        print(f"\n{sep}")
        print(f"👤 Pregunta: {question}")
        print(f"{sep}")
        print(f"\n🎬 Respuesta:\n")
        print(answer)
        if result.get("sources"):
            print(f"\n📎 Fuentes: {', '.join(result['sources'][:3])}")
        print(f"\n{sep}")

    return answer


# Instanciar memoria global de sesión
memory = ConversationMemory(max_turns=10)

print("✅ Memoria inicializada  |  max_turns =", memory.max_turns)
print("✅ Función ask() lista")
print("\nUso:")
print("   ask('¿Cuál es el argumento de Pulp Fiction?')")
print("   memory.show()   # ver historial")
print("   memory.clear()  # borrar historial")

✅ Memoria inicializada  |  max_turns = 10
✅ Función ask() lista

Uso:
   ask('¿Cuál es el argumento de Pulp Fiction?')
   memory.show()   # ver historial
   memory.clear()  # borrar historial


## Paso 8 — Ejemplos documentados

In [9]:
# Limpiar memoria antes de los ejemplos
memory.clear()

# ── Ejemplo 1: Argumento y personajes ─────────────────────────
print("\n" + "█" * 60)
print("  EJEMPLO 1 — Argumento y personajes")
print("█" * 60)
_ = ask(
    "¿Cuál es el argumento principal de Pulp Fiction "
    "y quiénes son sus personajes centrales?"
)

# ── Ejemplo 2: Ficha técnica ───────────────────────────────────
print("\n" + "█" * 60)
print("  EJEMPLO 2 — Ficha técnica")
print("█" * 60)
_ = ask(
    "Dame la ficha técnica de Kill Bill: "
    "director de fotografía, música, año de estreno y presupuesto."
)

# ── Ejemplo 3: Premios ─────────────────────────────────────────
print("\n" + "█" * 60)
print("  EJEMPLO 3 — Premios y reconocimientos")
print("█" * 60)
_ = ask(
    "¿Qué premios y nominaciones importantes ha recibido "
    "Quentin Tarantino a lo largo de su carrera?"
)

# ── Ejemplo 4: Estilo cinematográfico ──────────────────────────
print("\n" + "█" * 60)
print("  EJEMPLO 4 — Estilo cinematográfico")
print("█" * 60)
_ = ask(
    "¿Cuáles son las características del estilo cinematográfico "
    "de Tarantino? Menciona recursos narrativos y visuales recurrentes."
)

# ── Ejemplo 5: PRUEBA DE MEMORIA ───────────────────────────────
# Esta pregunta referencia explícitamente respuestas anteriores
# para demostrar que el agente mantiene el contexto entre turnos.
print("\n" + "█" * 60)
print("  EJEMPLO 5 — PRUEBA DE MEMORIA (referencia a turno anterior)")
print("█" * 60)
_ = ask(
    "De los premios que mencionaste antes, ¿cuál consideras "
    "el más importante para su trayectoria? "
    "¿Y cómo se relaciona con el estilo cinematográfico que describiste?"
)

# ── Ejemplo 6: Recepción crítica ───────────────────────────────
print("\n" + "█" * 60)
print("  EJEMPLO 6 — Recepción crítica")
print("█" * 60)
_ = ask(
    "¿Cómo fue recibida Inglourious Basterds "
    "por la crítica y el público? ¿Generó alguna controversia?"
)

# ── Ejemplo 7: Límite del sistema ──────────────────────────────
# El agente debe admitir que no tiene la información en lugar de inventar.
print("\n" + "█" * 60)
print("  EJEMPLO 7 — Límite del sistema (honestidad)")
print("█" * 60)
_ = ask(
    "¿Cuál es la opinión de Tarantino sobre el cine de Marvel "
    "y el streaming moderno?"
)

# -- Estado final de la memoria --------------------------------
print("\n" + "═" * 60)
print(f"📊 Memoria final: {len(memory.history) // 2} turnos registrados")
print("═" * 60)
memory.show()

🗑  Historial de memoria limpiado.

████████████████████████████████████████████████████████████
  EJEMPLO 1 — Argumento y personajes
████████████████████████████████████████████████████████████

────────────────────────────────────────────────────────────
👤 Pregunta: ¿Cuál es el argumento principal de Pulp Fiction y quiénes son sus personajes centrales?
────────────────────────────────────────────────────────────

🎬 Respuesta:

Según los documentos proporcionados, el argumento principal de *Pulp Fiction* se centra en la **narrativa no lineal** de varias historias entrelazadas, cuyos protagonistas son miembros del crimen organizado de Los Ángeles. La película se distingue por sus diálogos estilizados, la mezcla de humor y violencia, y un tono irónico (Fragmento 1).

Más específicamente, *Pulp Fiction* está narrada en un orden no cronológico y cuenta tres historias principales que se relacionan entre sí para formar una intriga. Esta intriga ha sido descrita como "una trama por episodios,

## Paso 9 — Chat interactivo

In [11]:
# Ejecuta esta celda para abrir el chat en tiempo real.
# Comandos especiales:
#   salir   → termina el bucle
#   limpiar → borra el historial de memoria
#   memoria → muestra el historial actual

print("🎬 CHAT INTERACTIVO — Experto en Tarantino")
print("=" * 55)
print("Comandos: salir | limpiar | memoria")
print("=" * 55)

while True:
    try:
        user_input = input("\n👤 Tú: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n\n👋 Sesión terminada.")
        break

    if not user_input:
        continue

    if user_input.lower() == "salir":
        print("\n👋 ¡Hasta luego! Buenas películas.")
        break
    elif user_input.lower() == "limpiar":
        memory.clear()
    elif user_input.lower() == "memoria":
        print(f"\n📊 Historial ({len(memory.history) // 2} turnos):")
        memory.show()
    else:
        ask(user_input)

🎬 CHAT INTERACTIVO — Experto en Tarantino
Comandos: salir | limpiar | memoria

👤 Tú: ¿En cuantas peliculas aparece Leonardo dicaprio?

────────────────────────────────────────────────────────────
👤 Pregunta: ¿En cuantas peliculas aparece Leonardo dicaprio?
────────────────────────────────────────────────────────────

🎬 Respuesta:

Según los fragmentos proporcionados, Leonardo DiCaprio aparece en **una** película de Quentin Tarantino:

*   **Once Upon a Time in Hollywood** (Érase una vez en Hollywood / Había una vez en Hollywood).

Los fragmentos indican que Leonardo DiCaprio protagonizó esta película de 2019, escrita y dirigida por Quentin Tarantino (Fragmento 3). También se menciona que DiCaprio iba a protagonizar "Quentin Tarantino's Manson Movie" (Fragmento 1), que es el mismo proyecto que finalmente se tituló *Once Upon a Time in Hollywood*.

📎 Fuentes: Once_Upon_a_Time_in_Hollywood.pdf (pág. 16), Kill_Bill__Volumen_2.pdf (pág. 1), Once_Upon_a_Time_in_Hollywood.pdf (pág. 0)

──────

In [9]:
# CELDA — Reinstalar streamlit en el python correcto
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "streamlit"], check=True)
print("Instalado en:", sys.executable)

Instalado en: /usr/bin/python3


In [10]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("cloudflared instalado")

cloudflared instalado


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA A — Instalar Streamlit y localtunnel                  ║
# ╚══════════════════════════════════════════════════════════════╝

# !pip install -q streamlit
# !npm install -g localtunnel


# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA B — Escribir app.py en disco                          ║
# ╚══════════════════════════════════════════════════════════════╝

app_code = """
import os
import streamlit as st
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

GEMINI_API_KEY  = os.environ.get("GOOGLE_API_KEY", "")
CHROMA_DIR      = "/content/chroma_db"
COLLECTION_NAME = "tarantino_expert"
TOP_K           = 5

SYSTEM_PROMPT = (
    "Eres un asistente experto en el cine de Quentin Tarantino. "
    "Basa tus respuestas UNICAMENTE en los fragmentos recuperados. "
    "Si no tienes la informacion, dilo claramente. "
    "Cita la fuente cuando sea posible. Responde siempre en espanol."
)

st.set_page_config(page_title="Experto Tarantino", layout="wide")
st.title("Experto en Cine de Tarantino")
st.caption("RAG · Gemini 2.5 Flash · ChromaDB · LangChain")

@st.cache_resource
def load_agent():
    emb = GoogleGenerativeAIEmbeddings(
        model     = "gemini-embedding-001",
        task_type = "retrieval_document",
    )
    vs = Chroma(
        collection_name    = COLLECTION_NAME,
        embedding_function = emb,
        persist_directory  = CHROMA_DIR,
    )
    ret = vs.as_retriever(
        search_type   = "mmr",
        search_kwargs = {"k": TOP_K, "fetch_k": TOP_K * 3},
    )
    llm = ChatGoogleGenerativeAI(
        model       = "gemini-2.5-flash",
        temperature = 0.2,
        convert_system_message_to_human = True,
    )
    return ret, llm

retriever, llm = load_agent()

with st.sidebar:
    st.header("Sobre el agente")
    st.markdown(
        "**Dominio:** Cine de Quentin Tarantino\\n\\n"
        "**Stack:**\\n"
        "- Gemini 2.5 Flash\\n"
        "- ChromaDB (vectorstore)\\n"
        "- LangChain RAG\\n\\n"
        "**Como funciona:**\\n"
        "1. Tu pregunta se convierte en embedding\\n"
        "2. Se recuperan los fragmentos mas relevantes\\n"
        "3. Gemini genera la respuesta con ese contexto"
    )
    st.divider()
    count = retriever.vectorstore._collection.count()
    st.metric("Chunks indexados", count)
    if st.button("Limpiar conversacion"):
        st.session_state.messages = []
        st.rerun()

if "messages" not in st.session_state:
    st.session_state.messages = []

# Mostrar historial
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Input con text_input + boton (evita el bug de JS de st.chat_input)
st.divider()
col1, col2 = st.columns([5, 1])
with col1:
    prompt = st.text_input(
        "Tu pregunta:",
        placeholder = "Pregunta sobre las peliculas de Tarantino...",
        label_visibility = "collapsed",
        key = "input_box",
    )
with col2:
    enviar = st.button("Enviar", use_container_width=True)

if enviar and prompt.strip():

    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.spinner("Consultando la base de conocimiento..."):

        docs = retriever.invoke(prompt)
        context_parts = []
        for i, d in enumerate(docs):
            src  = d.metadata.get("source", "N/A").split("/")[-1]
            page = d.metadata.get("page", "?")
            context_parts.append(
                f"[Fragmento {i+1} - {src}, pag. {page}]\\n{d.page_content}"
            )
        context = "\\n\\n".join(context_parts)

        history = []
        for m in st.session_state.messages[:-1]:
            if m["role"] == "user":
                history.append(HumanMessage(content=m["content"]))
            else:
                history.append(AIMessage(content=m["content"]))

        msgs = [
            SystemMessage(content=SYSTEM_PROMPT),
            *history,
            HumanMessage(content=(
                "CONTEXTO RECUPERADO:\\n" + context +
                "\\n\\n" + "=" * 40 + "\\n\\n" +
                "PREGUNTA: " + prompt
            )),
        ]
        answer = llm.invoke(msgs).content

    st.session_state.messages.append({"role": "assistant", "content": answer})

    with st.expander("Ver fragmentos recuperados"):
        for i, d in enumerate(docs):
            src  = d.metadata.get("source", "N/A").split("/")[-1]
            page = d.metadata.get("page", "?")
            st.caption(f"Fragmento {i+1} - {src}, pag. {page}")
            st.text(d.page_content[:300] + "...")
            st.divider()

    st.rerun()
"""

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py guardado en /content/app.py")


# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA C1 — Verificar PATH de streamlit                      ║
# ╚══════════════════════════════════════════════════════════════╝

import sys
import os
import subprocess

# Añadir rutas de binarios al PATH
sys.path.append("/usr/local/lib/python3.12/dist-packages")
os.environ["PATH"] += ":/usr/local/bin"

# Verificar que streamlit es localizable
result = subprocess.run(["which", "streamlit"], capture_output=True, text=True)
print(f"Streamlit encontrado en: {result.stdout.strip()}")


# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA C — Lanzar Streamlit + localtunnel                    ║
# ╚══════════════════════════════════════════════════════════════╝

# CELDA C — Lanzar Streamlit + cloudflared
import subprocess, threading, time, os, sys, re

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# 1. Lanzar Streamlit
def run_streamlit():
    subprocess.run([
        sys.executable, "-m", "streamlit",
        "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ], env=os.environ)

threading.Thread(target=run_streamlit, daemon=True).start()
print("Iniciando Streamlit...")
time.sleep(6)
print("Streamlit corriendo en puerto 8501")

# 2. Abrir tunel con cloudflared (no requiere cuenta ni token)
tunnel = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
)

print("Abriendo tunel cloudflared...")
time.sleep(5)

# Leer URL del tunel de los logs
for _ in range(30):
    line = tunnel.stdout.readline().decode("utf8", errors="ignore").strip()
    if "trycloudflare.com" in line:
        url = re.search(r'https://\S+\.trycloudflare\.com', line)
        if url:
            print(f"\nApp disponible en: {url.group()}")
            print("(No pide contrasena, abre directamente)")
            break

app.py guardado en /content/app.py
Streamlit encontrado en: /usr/local/bin/streamlit
Iniciando Streamlit...
Streamlit corriendo en puerto 8501
Abriendo tunel cloudflared...

App disponible en: https://platforms-pants-roof-player.trycloudflare.com
(No pide contrasena, abre directamente)
